# 🤖 Wordle RL Solver — Phase 1: PPO Training

Trains a PPO agent to play Wordle using the same 390-dim board encoding as the supervised model.

**Runtime:** T4 GPU recommended · ~2–4 hrs for 500k episodes

**Key features:**
- Action masking (only valid words per step)
- Curriculum learning (easy → hard)
- Reward shaping for constraint-consistent guesses
- Checkpointing to Google Drive
- Final benchmark vs supervised (3.46 avg, 100% win rate)

## 0. Setup & Installs

In [ ]:
!pip install -q gymnasium huggingface_hub

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/wordle_rl_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Drive mounted. Checkpoints will save to:', SAVE_DIR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import json
import random
import time
from collections import deque
from huggingface_hub import hf_hub_download

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Word Lists from HF Hub

In [ ]:
REPO_ID = 'sato2ru/wordle-solver'

answers_path = hf_hub_download(repo_id=REPO_ID, filename='answers.json')
allowed_path = hf_hub_download(repo_id=REPO_ID, filename='allowed.json')

with open(answers_path) as f:
    ANSWERS = json.load(f)
with open(allowed_path) as f:
    ALLOWED = json.load(f)

# Build word->index lookup for fast action encoding
WORD2IDX = {w: i for i, w in enumerate(ALLOWED)}

print(f'Answers: {len(ANSWERS)} | Allowed guesses: {len(ALLOWED)}')

## 2. Wordle Environment

In [ ]:
def get_pattern(guess: str, answer: str) -> tuple:
    """Returns (0,1,2) pattern. 0=grey, 1=yellow, 2=green."""
    pattern = [0] * 5
    answer_counts = {}
    # First pass: greens
    for i in range(5):
        if guess[i] == answer[i]:
            pattern[i] = 2
        else:
            answer_counts[answer[i]] = answer_counts.get(answer[i], 0) + 1
    # Second pass: yellows
    for i in range(5):
        if pattern[i] == 0 and guess[i] in answer_counts and answer_counts[guess[i]] > 0:
            pattern[i] = 1
            answer_counts[guess[i]] -= 1
    return tuple(pattern)


def filter_words(words: list, guess: str, pattern: tuple) -> list:
    """Filter word list to those consistent with the guess/pattern."""
    return [w for w in words if get_pattern(guess, w) == pattern]


def encode_board(history: list) -> np.ndarray:
    """Encode guess history to 390-dim binary vector (same as supervised model)."""
    vec = np.zeros(390, dtype=np.float32)
    for guess, pattern in history:
        for pos, (letter, state) in enumerate(zip(guess, pattern)):
            letter_idx = ord(letter) - ord('a')
            vec[letter_idx * 15 + pos * 3 + state] = 1.0
    return vec


class WordleEnv:
    """
    Custom Wordle environment.
    
    State:   390-dim board encoding
    Action:  index into ALLOWED word list (0..12971)
    Reward:  +10 win, -1/guess, -10 lose
    """
    MAX_GUESSES = 6

    def __init__(self, answer_pool=None, curriculum_k=None):
        """
        answer_pool: list of possible answers (default: all ANSWERS)
        curriculum_k: if set, randomly pick answer from a pool of k candidates
                      that share a common prefix — makes early training easier.
                      Set to None to disable curriculum.
        """
        self.answer_pool = answer_pool or ANSWERS
        self.curriculum_k = curriculum_k
        self.reset()

    def reset(self):
        if self.curriculum_k:
            # Curriculum: pick a random anchor, then sample nearby answers
            anchor = random.choice(self.answer_pool)
            pool = [w for w in self.answer_pool if w[0] == anchor[0]]
            self.answer = random.choice(pool[:self.curriculum_k] if len(pool) >= self.curriculum_k else pool)
        else:
            self.answer = random.choice(self.answer_pool)
        self.history = []  # list of (guess, pattern) tuples
        self.possible = list(ANSWERS)  # words still consistent with clues
        self.guessed = set()
        self.done = False
        return encode_board(self.history)

    def valid_action_mask(self) -> np.ndarray:
        """Binary mask over ALLOWED: 1 if word hasn't been guessed yet."""
        mask = np.ones(len(ALLOWED), dtype=bool)
        for w in self.guessed:
            mask[WORD2IDX[w]] = False
        return mask

    def step(self, action: int):
        assert not self.done, 'Episode finished — call reset()'
        guess = ALLOWED[action]
        pattern = get_pattern(guess, self.answer)
        self.history.append((guess, pattern))
        self.guessed.add(guess)
        self.possible = filter_words(self.possible, guess, pattern)

        won = all(p == 2 for p in pattern)
        n_guesses = len(self.history)

        # Reward shaping
        reward = -1.0  # base cost per guess
        if won:
            reward += 10.0
            self.done = True
        elif n_guesses >= self.MAX_GUESSES:
            reward -= 10.0
            self.done = True
        else:
            # Small bonus if guess is consistent with remaining constraints
            if guess in self.possible:
                reward += 0.1

        state = encode_board(self.history)
        info = {'won': won, 'n_guesses': n_guesses, 'answer': self.answer, 'possible_left': len(self.possible)}
        return state, reward, self.done, info


# Quick sanity check
env = WordleEnv()
s = env.reset()
print(f'State shape: {s.shape}  |  Answer (hidden): {env.answer}')
mask = env.valid_action_mask()
s2, r, done, info = env.step(np.argmax(mask))  # guess first valid word
print(f'Step result — reward: {r:.1f}, done: {done}, info: {info}')

## 3. PPO Policy Network

In [ ]:
class WordlePolicy(nn.Module):
    """
    Actor-Critic network.
    Same encoder as supervised WordleNet for fair comparison.
    Policy head: 12972-dim softmax
    Value head:  scalar
    """
    def __init__(self, input_dim=390, hidden=512, action_dim=12972):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(256, action_dim)
        self.value_head  = nn.Linear(256, 1)

    def forward(self, x, action_mask=None):
        """
        x:           (B, 390) float tensor
        action_mask: (B, 12972) bool tensor — True = valid action
        Returns: action_probs (B, 12972), values (B,)
        """
        feat = self.encoder(x)
        logits = self.policy_head(feat)        # (B, 12972)
        if action_mask is not None:
            # Fill invalid actions with -inf so they get ~0 probability
            logits = logits.masked_fill(~action_mask, float('-inf'))
        probs  = F.softmax(logits, dim=-1)
        values = self.value_head(feat).squeeze(-1)  # (B,)
        return probs, values

    def get_action(self, state_np: np.ndarray, mask_np: np.ndarray):
        """Sample an action during rollout (no grad)."""
        self.eval()
        with torch.no_grad():
            x    = torch.FloatTensor(state_np).unsqueeze(0).to(device)
            mask = torch.BoolTensor(mask_np).unsqueeze(0).to(device)
            probs, value = self(x, mask)
            dist   = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
        return action.item(), log_prob.item(), value.item()


policy = WordlePolicy().to(device)
total_params = sum(p.numel() for p in policy.parameters())
print(f'Policy network: {total_params:,} parameters')

## 4. PPO Agent

In [ ]:
class PPOAgent:
    def __init__(
        self,
        policy: WordlePolicy,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_eps=0.2,
        entropy_coef=0.01,
        value_coef=0.5,
        max_grad_norm=0.5,
        ppo_epochs=4,
        minibatch_size=256,
    ):
        self.policy      = policy
        self.optimizer   = optim.Adam(policy.parameters(), lr=lr)
        self.gamma       = gamma
        self.gae_lambda  = gae_lambda
        self.clip_eps    = clip_eps
        self.entropy_coef = entropy_coef
        self.value_coef  = value_coef
        self.max_grad_norm = max_grad_norm
        self.ppo_epochs  = ppo_epochs
        self.minibatch_size = minibatch_size

    def _compute_gae(self, rewards, values, dones):
        """Generalized Advantage Estimation."""
        advantages = np.zeros_like(rewards, dtype=np.float32)
        last_gae = 0.0
        T = len(rewards)
        # Bootstrap value for the last step
        next_value = values[-1] if not dones[-1] else 0.0
        for t in reversed(range(T)):
            if t == T - 1:
                next_val = next_value
                next_done = dones[-1]
            else:
                next_val = values[t + 1]
                next_done = dones[t + 1]
            delta = rewards[t] + self.gamma * next_val * (1 - next_done) - values[t]
            last_gae = delta + self.gamma * self.gae_lambda * (1 - next_done) * last_gae
            advantages[t] = last_gae
        returns = advantages + values
        return advantages, returns

    def update(self, rollout: dict) -> dict:
        """Run PPO update on collected rollout data."""
        states    = torch.FloatTensor(rollout['states']).to(device)
        actions   = torch.LongTensor(rollout['actions']).to(device)
        old_lps   = torch.FloatTensor(rollout['log_probs']).to(device)
        masks     = torch.BoolTensor(rollout['masks']).to(device)
        advantages= torch.FloatTensor(rollout['advantages']).to(device)
        returns   = torch.FloatTensor(rollout['returns']).to(device)

        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        N = states.shape[0]
        indices = np.arange(N)
        metrics = {'policy_loss': [], 'value_loss': [], 'entropy': []}

        self.policy.train()
        for _ in range(self.ppo_epochs):
            np.random.shuffle(indices)
            for start in range(0, N, self.minibatch_size):
                mb_idx = indices[start:start + self.minibatch_size]
                mb_idx = torch.LongTensor(mb_idx).to(device)

                probs, values = self.policy(states[mb_idx], masks[mb_idx])
                dist = torch.distributions.Categorical(probs)
                new_lps  = dist.log_prob(actions[mb_idx])
                entropy  = dist.entropy().mean()

                ratio = (new_lps - old_lps[mb_idx]).exp()
                adv   = advantages[mb_idx]

                # Clipped surrogate objective
                surr1 = ratio * adv
                surr2 = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * adv
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss (clipped)
                value_loss = F.mse_loss(values, returns[mb_idx])

                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.optimizer.step()

                metrics['policy_loss'].append(policy_loss.item())
                metrics['value_loss'].append(value_loss.item())
                metrics['entropy'].append(entropy.item())

        return {k: np.mean(v) for k, v in metrics.items()}


agent = PPOAgent(policy)
print('PPO Agent initialized.')

## 5. Training Loop

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
TOTAL_EPISODES     = 500_000   # increase to 1M for better convergence
ROLLOUT_EPISODES   = 512       # episodes per rollout batch
CHECKPOINT_EVERY   = 50_000    # save every N episodes
LOG_EVERY          = 2_000     # print stats every N episodes

# Curriculum schedule: start easy (small answer pool), end full difficulty
# Set to None to skip curriculum entirely
CURRICULUM_SCHEDULE = [
    (0,       50),    # episodes 0–50k: answer pool limited to 50 words
    (50_000,  200),
    (150_000, 500),
    (300_000, None),  # full difficulty from 300k onwards
]


def get_curriculum_pool(episode: int) -> list:
    """Returns answer pool for current curriculum stage."""
    k = None
    for ep_threshold, pool_size in CURRICULUM_SCHEDULE:
        if episode >= ep_threshold:
            k = pool_size
    if k is None:
        return ANSWERS
    # Deterministic pool for reproducibility at each stage
    return ANSWERS[:k]


def collect_rollout(agent, n_episodes: int, episode_offset: int) -> dict:
    """Collect n_episodes of experience."""
    rollout = {'states': [], 'actions': [], 'log_probs': [], 'rewards': [],
               'values': [], 'dones': [], 'masks': []}
    ep_rewards, ep_lengths, wins = [], [], []

    pool = get_curriculum_pool(episode_offset)
    env = WordleEnv(answer_pool=pool)

    for _ in range(n_episodes):
        state = env.reset()
        ep_r = 0.0
        while True:
            mask  = env.valid_action_mask()
            action, log_prob, value = agent.policy.get_action(state, mask)

            rollout['states'].append(state)
            rollout['actions'].append(action)
            rollout['log_probs'].append(log_prob)
            rollout['values'].append(value)
            rollout['masks'].append(mask)

            next_state, reward, done, info = env.step(action)
            rollout['rewards'].append(reward)
            rollout['dones'].append(float(done))

            ep_r   += reward
            state   = next_state
            if done:
                ep_rewards.append(ep_r)
                ep_lengths.append(info['n_guesses'])
                wins.append(int(info['won']))
                break

    # Convert to arrays
    for k in ['states', 'masks']:
        rollout[k] = np.array(rollout[k])
    for k in ['actions', 'log_probs', 'rewards', 'values', 'dones']:
        rollout[k] = np.array(rollout[k], dtype=np.float32)
    rollout['actions'] = rollout['actions'].astype(np.int64)

    advantages, returns = agent._compute_gae(
        rollout['rewards'], rollout['values'], rollout['dones'])
    rollout['advantages'] = advantages
    rollout['returns']    = returns

    stats = {'mean_reward': np.mean(ep_rewards),
             'mean_length': np.mean(ep_lengths),
             'win_rate':    np.mean(wins)}
    return rollout, stats


# ── Main Training Loop ───────────────────────────────────────────────────────
print('Starting training...')
print(f'Total episodes: {TOTAL_EPISODES:,} | Rollout batch: {ROLLOUT_EPISODES}')

total_episodes = 0
history_log    = []
t0             = time.time()

while total_episodes < TOTAL_EPISODES:
    rollout, stats = collect_rollout(agent, ROLLOUT_EPISODES, total_episodes)
    loss_metrics   = agent.update(rollout)
    total_episodes += ROLLOUT_EPISODES

    log_entry = {**stats, **loss_metrics, 'episodes': total_episodes}
    history_log.append(log_entry)

    if total_episodes % LOG_EVERY < ROLLOUT_EPISODES:
        elapsed = time.time() - t0
        stage = [k for ep, k in CURRICULUM_SCHEDULE if total_episodes >= ep][-1]
        stage_str = f'{stage}' if stage else 'FULL'
        print(f"[{total_episodes:>7,}] "
              f"win={stats['win_rate']:.1%}  "
              f"avg_guesses={stats['mean_length']:.2f}  "
              f"reward={stats['mean_reward']:.2f}  "
              f"entropy={loss_metrics['entropy']:.3f}  "
              f"stage={stage_str}  "
              f"elapsed={elapsed/60:.1f}m")

    if total_episodes % CHECKPOINT_EVERY < ROLLOUT_EPISODES:
        ckpt_path = f'{SAVE_DIR}/checkpoint_{total_episodes}.pt'
        torch.save({
            'episodes':    total_episodes,
            'model_state': policy.state_dict(),
            'optim_state': agent.optimizer.state_dict(),
            'history':     history_log,
        }, ckpt_path)
        print(f'  💾 Checkpoint saved: {ckpt_path}')

print(f'\nTraining complete! {total_episodes:,} episodes in {(time.time()-t0)/60:.1f} min')

## 6. Full Benchmark (All 2,315 Answers)

In [ ]:
def run_benchmark(policy: WordlePolicy, greedy=True) -> dict:
    """
    Run all 2,315 Wordle games and report stats.
    greedy=True: always pick argmax action (no sampling noise)
    """
    policy.eval()
    results = []
    openers = {}

    for answer in ANSWERS:
        env = WordleEnv(answer_pool=[answer])  # fixed answer
        state = env.reset()
        while True:
            mask = env.valid_action_mask()
            with torch.no_grad():
                x     = torch.FloatTensor(state).unsqueeze(0).to(device)
                m     = torch.BoolTensor(mask).unsqueeze(0).to(device)
                probs, _ = policy(x, m)
                action = probs.argmax(dim=-1).item() if greedy else \
                         torch.distributions.Categorical(probs).sample().item()
            state, _, done, info = env.step(action)
            if done:
                results.append({'answer': answer, 'won': info['won'],
                                'n_guesses': info['n_guesses']})
                opener = env.history[0][0]
                openers[opener] = openers.get(opener, 0) + 1
                break

    wins = [r for r in results if r['won']]
    losses = [r for r in results if not r['won']]
    guess_dist = {}
    for r in wins:
        guess_dist[r['n_guesses']] = guess_dist.get(r['n_guesses'], 0) + 1

    top_openers = sorted(openers.items(), key=lambda x: -x[1])[:5]

    return {
        'win_rate':     len(wins) / len(results),
        'avg_guesses':  np.mean([r['n_guesses'] for r in wins]) if wins else 0,
        'guess_dist':   guess_dist,
        'top_openers':  top_openers,
        'losses':       [r['answer'] for r in losses],
    }


print('Running full benchmark on all 2,315 answers (greedy)...')
benchmark = run_benchmark(policy, greedy=True)

print(f"\n{'='*50}")
print(f"RL Model Results")
print(f"{'='*50}")
print(f"Win rate:     {benchmark['win_rate']:.1%}  (supervised: 100%)")
print(f"Avg guesses:  {benchmark['avg_guesses']:.3f}  (supervised: 3.46)")
print(f"\nGuess distribution:")
for g in sorted(benchmark['guess_dist']):
    bar = '█' * (benchmark['guess_dist'][g] // 10)
    print(f"  {g} guesses: {benchmark['guess_dist'][g]:>4} games  {bar}")
print(f"\nTop opener words:")
for word, count in benchmark['top_openers']:
    print(f"  {word}: {count} games")
if benchmark['losses']:
    print(f"\nFailed games ({len(benchmark['losses'])}): {benchmark['losses'][:20]}")

## 7. Save Final Model & Upload to HF Hub

In [ ]:
import os
from huggingface_hub import HfApi, upload_file

# ── Save model locally ───────────────────────────────────────────────────────
FINAL_WEIGHTS = '/content/rl_model_weights.pt'
FINAL_CONFIG  = '/content/rl_config.json'

torch.save(policy.state_dict(), FINAL_WEIGHTS)

rl_config = {
    'input_dim':     390,
    'output_dim':    12972,
    'hidden':        512,
    'algorithm':     'PPO',
    'total_episodes': TOTAL_EPISODES,
    'win_rate':      benchmark['win_rate'],
    'avg_guesses':   benchmark['avg_guesses'],
    'top_openers':   benchmark['top_openers'],
    'architecture':  '390->512->512->256->12972 (policy) + 1 (value)',
}
with open(FINAL_CONFIG, 'w') as f:
    json.dump(rl_config, f, indent=2)

print('Model weights and config saved locally.')
print('config:', rl_config)

# ── Upload to HF Hub ─────────────────────────────────────────────────────────
# Option A: same repo as supervised, new files
# Option B: new repo sato2ru/wordle-solver-rl
# Uncomment and run whichever you prefer:

# HF_TOKEN = os.environ.get('HF_TOKEN') or 'hf_...'  # <-- paste your token here (DO NOT COMMIT)
# TARGET_REPO = 'sato2ru/wordle-solver'   # or 'sato2ru/wordle-solver-rl'

# api = HfApi(token=HF_TOKEN)
# api.upload_file(path_or_fileobj=FINAL_WEIGHTS, path_in_repo='rl_model_weights.pt', repo_id=TARGET_REPO)
# api.upload_file(path_or_fileobj=FINAL_CONFIG,  path_in_repo='rl_config.json',      repo_id=TARGET_REPO)
# print(f'Uploaded to https://huggingface.co/{TARGET_REPO}')

print('\n✅ Phase 1 complete! Ready for Phase 2 (HF upload) and Phase 3 (backend update).')

## 8. (Optional) Training Curve Visualization

In [ ]:
import matplotlib.pyplot as plt

episodes  = [h['episodes'] for h in history_log]
win_rates = [h['win_rate'] for h in history_log]
avg_lens  = [h['mean_length'] for h in history_log]
entropies = [h['entropy'] for h in history_log]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Wordle RL Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(episodes, win_rates, color='green')
axes[0].axhline(1.0, color='gray', linestyle='--', label='Supervised (100%)')
axes[0].set_title('Win Rate'); axes[0].set_xlabel('Episodes'); axes[0].legend()

axes[1].plot(episodes, avg_lens, color='blue')
axes[1].axhline(3.46, color='gray', linestyle='--', label='Supervised (3.46)')
axes[1].set_title('Avg Guesses (wins only)'); axes[1].set_xlabel('Episodes'); axes[1].legend()

axes[2].plot(episodes, entropies, color='orange')
axes[2].set_title('Policy Entropy'); axes[2].set_xlabel('Episodes')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print('Curves saved to Drive.')